# Subtask 1: Dimensional Aspect Sentiment Regression (DimASR)
Given text and aspects, predict VA scores for each aspect.

**Input**: Text + List of aspects
**Output**: VA scores (valence#arousal) for each aspect

In [1]:
# Install required packages
!pip install -q transformers datasets torch accelerate

In [2]:
import json
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup #,AdamW
from torch.optim import AdamW
from torch import nn
from tqdm.auto import tqdm
import os
import zipfile
from typing import List, Dict, Tuple
import warnings
import shutil
warnings.filterwarnings('ignore')

In [3]:
# Configuration
class Config:
    # Paths (adjust these to your Kaggle dataset paths)
    TRAIN_ZIP = '/kaggle/input/dimasr/subtask1'
    DATA_DIR = './subtask1_data'
    OUTPUT_DIR = './subtask1_outputs'
    CHECKPOINT_DIR = './subtask1_checkpoints'
    
    # Model
    MODEL_NAME = 'bert-base-uncased'  # Using BERT for regression
    MAX_LENGTH = 128
    HIDDEN_DIM = 256
    DROPOUT = 0.3
    
    # Training
    BATCH_SIZE = 32
    EPOCHS = 20
    LEARNING_RATE = 2e-5
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    GRADIENT_CLIP = 1.0
    
    # VA normalization
    VA_MIN = 1.0
    VA_MAX = 9.0
    
    # Device
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Random seed
    SEED = 42

config = Config()
os.makedirs(config.DATA_DIR, exist_ok=True)
os.makedirs(config.TRAIN_ZIP, exist_ok=True)
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
files_to_copy = ["eng_laptop_train_alltasks.jsonl", "eng_laptop_dev_task1.jsonl",
            "eng_laptop_test_task1.jsonl", "eng_restaurant_train_alltasks.jsonl",
            "eng_restaurant_dev_task1.jsonl", "eng_restaurant_test_task1.jsonl"]
TRAIN_ZIP = '/kaggle/input/dimasr/subtask1'
DATA_DIR = './subtask1_data'
for fname in files_to_copy:
    src = os.path.join(TRAIN_ZIP, fname)
    dst = os.path.join(DATA_DIR, fname)
    shutil.copy(src, dst)
    print(f"Copied {fname} → {dst}")
# Set seed
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)

print(f"Device: {config.DEVICE}")

Copied eng_laptop_train_alltasks.jsonl → ./subtask1_data/eng_laptop_train_alltasks.jsonl
Copied eng_laptop_dev_task1.jsonl → ./subtask1_data/eng_laptop_dev_task1.jsonl
Copied eng_laptop_test_task1.jsonl → ./subtask1_data/eng_laptop_test_task1.jsonl
Copied eng_restaurant_train_alltasks.jsonl → ./subtask1_data/eng_restaurant_train_alltasks.jsonl
Copied eng_restaurant_dev_task1.jsonl → ./subtask1_data/eng_restaurant_dev_task1.jsonl
Copied eng_restaurant_test_task1.jsonl → ./subtask1_data/eng_restaurant_test_task1.jsonl
Device: cuda


In [4]:
# Data loading functions
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

def normalize_va(value):
    """Normalize VA from [1,9] to [0,1]"""
    return (value - config.VA_MIN) / (config.VA_MAX - config.VA_MIN)

def denormalize_va(value):
    """Denormalize VA from [0,1] to [1,9]"""
    return value * (config.VA_MAX - config.VA_MIN) + config.VA_MIN

def parse_va(va_string):
    """Parse VA string 'V.VV#A.AA' to (valence, arousal)"""
    v, a = va_string.split('#')
    return float(v), float(a)

def format_va(valence, arousal):
    """Format VA to 'V.VV#A.AA' with proper clipping and rounding"""
    v = np.clip(valence, config.VA_MIN, config.VA_MAX)
    a = np.clip(arousal, config.VA_MIN, config.VA_MAX)
    return f"{v:.2f}#{a:.2f}"

In [5]:
# Dataset class
class DimASRDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # Flatten data: one sample per aspect
        self.samples = []
        for item in data:
            text = item['Text']
            
            # Handle both formats: 'Aspect_VA' (dev/test) and 'Quadruplet' (train)
            aspect_list = item.get('Aspect_VA', item.get('Quadruplet', []))
            
            for aspect_item in aspect_list:
                aspect = aspect_item['Aspect']
                va = aspect_item['VA']
                v, a = parse_va(va)
                self.samples.append({
                    'id': item['ID'],
                    'text': text,
                    'aspect': aspect,
                    'valence': normalize_va(v),
                    'arousal': normalize_va(a)
                })
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Format: [CLS] text [SEP] aspect [SEP]
        encoding = self.tokenizer(
            sample['text'],
            sample['aspect'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'valence': torch.tensor(sample['valence'], dtype=torch.float),
            'arousal': torch.tensor(sample['arousal'], dtype=torch.float),
            'id': sample['id'],
            'text': sample['text'],
            'aspect': sample['aspect']
        }

# Test dataset (no labels)
class DimASRTestDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # Flatten data: one sample per aspect
        self.samples = []
        for item in data:
            text = item['Text']
            aspects = item.get('Aspect', [])
            for aspect in aspects:
                self.samples.append({
                    'id': item['ID'],
                    'text': text,
                    'aspect': aspect
                })
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        encoding = self.tokenizer(
            sample['text'],
            sample['aspect'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'id': sample['id'],
            'text': sample['text'],
            'aspect': sample['aspect']
        }

In [6]:
# Model architecture
class VARegressorModel(nn.Module):
    def __init__(self, base_model_name, hidden_dim, dropout):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        self.hidden_dim = hidden_dim
        
        encoder_dim = self.encoder.config.hidden_size
        
        # VA regression head
        self.va_head = nn.Sequential(
            nn.Linear(encoder_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 2)  # valence, arousal
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        
        va_pred = self.va_head(cls_output)  # [batch, 2]
        va_pred = torch.sigmoid(va_pred)  # Constrain to [0,1]
        
        return va_pred

In [7]:
# Training function
def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(dataloader, desc='Training')
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        valence = batch['valence'].to(device)
        arousal = batch['arousal'].to(device)
        
        optimizer.zero_grad()
        
        va_pred = model(input_ids, attention_mask)
        
        # Smooth L1 loss for regression
        valence_loss = nn.functional.smooth_l1_loss(va_pred[:, 0], valence)
        arousal_loss = nn.functional.smooth_l1_loss(va_pred[:, 1], arousal)
        loss = valence_loss + arousal_loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})
    
    return total_loss / len(dataloader)

In [8]:
# Evaluation function
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    all_v_pred, all_v_true = [], []
    all_a_pred, all_a_true = [], []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            valence = batch['valence'].to(device)
            arousal = batch['arousal'].to(device)
            
            va_pred = model(input_ids, attention_mask)
            
            valence_loss = nn.functional.smooth_l1_loss(va_pred[:, 0], valence)
            arousal_loss = nn.functional.smooth_l1_loss(va_pred[:, 1], arousal)
            loss = valence_loss + arousal_loss
            total_loss += loss.item()
            
            # Denormalize for RMSE calculation
            v_pred = denormalize_va(va_pred[:, 0].cpu().numpy())
            a_pred = denormalize_va(va_pred[:, 1].cpu().numpy())
            v_true = denormalize_va(valence.cpu().numpy())
            a_true = denormalize_va(arousal.cpu().numpy())
            
            all_v_pred.extend(v_pred)
            all_v_true.extend(v_true)
            all_a_pred.extend(a_pred)
            all_a_true.extend(a_true)
    
    # Calculate RMSE
    v_rmse = np.sqrt(np.mean((np.array(all_v_pred) - np.array(all_v_true)) ** 2))
    a_rmse = np.sqrt(np.mean((np.array(all_a_pred) - np.array(all_a_true)) ** 2))
    avg_rmse = (v_rmse + a_rmse) / 2
    
    return total_loss / len(dataloader), v_rmse, a_rmse, avg_rmse

In [9]:
# Load data
print("Loading data...")
train_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_train_alltasks.jsonl')
train_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_train_alltasks.jsonl')
dev_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_dev_task1.jsonl')
dev_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_dev_task1.jsonl')

# Combine domains
train_data = train_rest + train_laptop
dev_data = dev_rest + dev_laptop

print(f"Train samples: {len(train_data)}")
print(f"Dev samples: {len(dev_data)}")

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

# Create datasets
train_dataset = DimASRDataset(train_data, tokenizer, config.MAX_LENGTH)
dev_dataset = DimASRDataset(dev_data, tokenizer, config.MAX_LENGTH)

print(f"Train aspect-level samples: {len(train_dataset)}")
print(f"Dev aspect-level samples: {len(dev_dataset)}")

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=config.BATCH_SIZE)

Loading data...
Train samples: 6360
Dev samples: 400


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train aspect-level samples: 9432
Dev aspect-level samples: 615


In [10]:
# Initialize model
print("Initializing model...")
model = VARegressorModel(
    config.MODEL_NAME,
    config.HIDDEN_DIM,
    config.DROPOUT
).to(config.DEVICE)

# Optimizer and scheduler
optimizer = AdamW(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

total_steps = len(train_loader) * config.EPOCHS
warmup_steps = int(total_steps * config.WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")

Initializing model...


2026-02-08 11:53:37.172104: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770551617.357680      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770551617.409134      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770551617.886872      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770551617.886914      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770551617.886917      55 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Total training steps: 5900
Warmup steps: 590


In [11]:
# Training loop
best_rmse = float('inf')
patience = 3  # Stop if no improvement for 3 epochs
patience_counter = 0
history = {'train_loss': [], 'dev_loss': [], 'dev_rmse': []}

print("\nStarting training...")
for epoch in range(config.EPOCHS):
    print(f"\nEpoch {epoch + 1}/{config.EPOCHS}")
    
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, config.DEVICE)
    dev_loss, v_rmse, a_rmse, avg_rmse = evaluate(model, dev_loader, config.DEVICE)
    
    history['train_loss'].append(train_loss)
    history['dev_loss'].append(dev_loss)
    history['dev_rmse'].append(avg_rmse)
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Dev Loss: {dev_loss:.4f}")
    print(f"Dev V-RMSE: {v_rmse:.4f}, A-RMSE: {a_rmse:.4f}, Avg-RMSE: {avg_rmse:.4f}")
    
    # Save best model
    if avg_rmse < best_rmse:
        best_rmse = avg_rmse
        patience_counter = 0  # Reset counter
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'rmse': avg_rmse,
        }, f'{config.CHECKPOINT_DIR}/best_model.pt')
        print(f"✓ Saved best model (RMSE: {avg_rmse:.4f})")
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{patience})")
        
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch + 1}")
            break

print(f"\nTraining completed. Best RMSE: {best_rmse:.4f}")


Starting training...

Epoch 1/20


Training:   0%|          | 0/295 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Train Loss: 0.0380
Dev Loss: 0.0141
Dev V-RMSE: 1.0820, A-RMSE: 0.8238, Avg-RMSE: 0.9529
✓ Saved best model (RMSE: 0.9529)

Epoch 2/20


Training:   0%|          | 0/295 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Train Loss: 0.0051
Dev Loss: 0.0086
Dev V-RMSE: 0.7377, A-RMSE: 0.7492, Avg-RMSE: 0.7434
No improvement (2/3)

Epoch 8/20


Training:   0%|          | 0/295 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Train Loss: 0.0044
Dev Loss: 0.0098
Dev V-RMSE: 0.7667, A-RMSE: 0.8123, Avg-RMSE: 0.7895
No improvement (3/3)

Early stopping at epoch 8

Training completed. Best RMSE: 0.7412


In [12]:
# Prediction function
def predict(model, dataloader, device):
    model.eval()
    predictions = {}
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Predicting'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            ids = batch['id']
            aspects = batch['aspect']
            
            va_pred = model(input_ids, attention_mask)
            
            # Denormalize
            v_pred = denormalize_va(va_pred[:, 0].cpu().numpy())
            a_pred = denormalize_va(va_pred[:, 1].cpu().numpy())
            
            # Group by ID
            for i, (id_, aspect, v, a) in enumerate(zip(ids, aspects, v_pred, a_pred)):
                if id_ not in predictions:
                    predictions[id_] = []
                predictions[id_].append({
                    'Aspect': aspect,
                    'VA': format_va(v, a)
                })
    
    return predictions

def save_predictions(predictions, original_data, output_path):
    """Save predictions in JSONL format matching original structure"""
    with open(output_path, 'w', encoding='utf-8') as f:
        for item in original_data:
            id_ = item['ID']
            output = {
                'ID': id_,
                'Aspect_VA': predictions.get(id_, [])
            }
            f.write(json.dumps(output) + '\n')
    print(f"Predictions saved to {output_path}")

In [13]:
# Load best model for inference
print("Loading best model...")
checkpoint = torch.load(f'{config.CHECKPOINT_DIR}/best_model.pt',weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded model from epoch {checkpoint['epoch']} with RMSE {checkpoint['rmse']:.4f}")

Loading best model...
Loaded model from epoch 4 with RMSE 0.7412


In [15]:
# Generate predictions for test sets
print("\nGenerating predictions for test sets...")

# Restaurant test
test_rest_data = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_test_task1.jsonl')
test_rest_dataset = DimASRTestDataset(test_rest_data, tokenizer, config.MAX_LENGTH)
test_rest_loader = DataLoader(test_rest_dataset, batch_size=config.BATCH_SIZE)
predictions_rest = predict(model, test_rest_loader, config.DEVICE)
save_predictions(predictions_rest, test_rest_data, f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl')

# Laptop test
test_laptop_data = load_jsonl(f'{config.DATA_DIR}/eng_laptop_test_task1.jsonl')
test_laptop_dataset = DimASRTestDataset(test_laptop_data, tokenizer, config.MAX_LENGTH)
test_laptop_loader = DataLoader(test_laptop_dataset, batch_size=config.BATCH_SIZE)
predictions_laptop = predict(model, test_laptop_loader, config.DEVICE)
save_predictions(predictions_laptop, test_laptop_data, f'{config.OUTPUT_DIR}/laptop_test_predictions.jsonl')

print("\nAll predictions completed!")


Generating predictions for test sets...


Predicting:   0%|          | 0/47 [00:00<?, ?it/s]

Predictions saved to ./subtask1_outputs/restaurant_test_predictions.jsonl


Predicting:   0%|          | 0/45 [00:00<?, ?it/s]

Predictions saved to ./subtask1_outputs/laptop_test_predictions.jsonl

All predictions completed!


In [22]:
# Sample predictions with original text
print("\nSample predictions:")
print("="*80)

# Load the CORRECT test file (not dev file!)
test_rest_data = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_test_task1.jsonl')

with open(f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            pred = json.loads(line)
            print(f"\n--- Sample {i+1} ---")
            print(f"Text: {test_rest_data[i]['Text']}")
            print(f"ID: {pred['ID']}")
            for aspect_va in pred['Aspect_VA']:
                print(f"  Aspect: {aspect_va['Aspect']}")
                print(f"  VA Scores: {aspect_va['VA']}")
            print("-" * 50)
print("="*80)


Sample predictions:

--- Sample 1 ---
Text: A friend suggested this cafe for a lunch date a while back and I cannot stay away
ID: rest26_aspect_va_test_1
  Aspect: cafe
  VA Scores: 5.78#5.88
--------------------------------------------------

--- Sample 2 ---
Text: The beer selection is second to none , but this time we had one drink and decided to spend our money elsewhere for dinner
ID: rest26_aspect_va_test_2
  Aspect: beer selection
  VA Scores: 5.37#5.60
--------------------------------------------------

--- Sample 3 ---
Text: It was pretty bland for my liking - in complete contrast to the sausage and pepper pizza - and the pepperoni was pretty sparse
ID: rest26_aspect_va_test_3
  Aspect: pepper pizza
  VA Scores: 3.77#5.96
  Aspect: pepperoni
  VA Scores: 3.97#5.78
  Aspect: sausage
  VA Scores: 3.72#5.93
--------------------------------------------------

--- Sample 4 ---
Text: Half price beer during a generous happy hour , a huge deck that welcomes dogs , and delicious key l